# NB2｜前處理：把難看的光譜變成能判讀的光譜

**食品分析｜拉曼光譜與 RamanSPy 入門系列（第 2 本，共 5 本）**

在食品分析裡，**前處理沒做好，後面所有結果都是垃圾**。這一本一步一步拆解 RamanSPy 的五個前處理步驟。

---
### 這一本你會學到
- 逐步做完裁切、去尖峰、平滑、基線校正、歸一化
- 理解每一步「解決什麼問題」「做過頭會怎樣」
- 用 `rp.preprocessing.Pipeline` 把五步驟串成一條生產線
- 一次處理 60 個樣品

> 💡 **完全沒寫過程式也沒關係。** 你只要做三件事：
> 1. 用滑鼠點每一格左邊的 ▶ 播放鍵（或按 `Shift + Enter`）
> 2. 看下面跑出來的圖和數字
> 3. 遇到 `# 👉 換你做` 的地方，照提示改一個數字或一個字，再跑一次


In [ ]:
# ===== 第一次執行請先跑這一格（大約 1 分鐘）=====
# 在 Google Colab 上，套件不是永久安裝的，每次重開都要跑一次。
!pip install -q ramanspy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ramanspy as rp

# 讓圖上的中文正常顯示（Colab 用）
!wget -q -O TaipeiSans.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_ 2>/dev/null
import matplotlib
try:
    matplotlib.font_manager.fontManager.addfont("TaipeiSans.ttf")
    matplotlib.rc("font", family="Taipei Sans TC Beta")
except Exception:
    pass
matplotlib.rcParams["axes.unicode_minus"] = False

print("準備完成！")


In [ ]:
# ===== 資料載入設定 =====
# 這一行由老師部署時自動填入正確的 GitHub 網址，學生不用改。
DATA_BASE = "https://raw.githubusercontent.com/Tai-ShengYeh/Tai-ShengYeh.github.io/main/ramanspy-food-analysis/data/"

# 若你把 CSV 直接上傳到 Colab 左側「檔案」，把上面那行改成： DATA_BASE = ""
# 若你在自己電腦跑，且 data 資料夾就在旁邊，改成：       DATA_BASE = "data/"

def load_spectra(filename):
    """讀 CSV → 回傳 (樣品資訊表 meta, ramanspy 光譜物件 spectra)"""
    df = pd.read_csv(DATA_BASE + filename)
    meta_cols = [c for c in df.columns if not c.replace(".", "", 1).isdigit()]
    axis = np.array([float(c) for c in df.columns if c not in meta_cols])
    spectra = rp.SpectralContainer(df.drop(columns=meta_cols).values, axis)
    return df[meta_cols].reset_index(drop=True), spectra

print("load_spectra() 已定義，資料來源：", DATA_BASE or "（Colab 本機檔案）")


In [ ]:
raw = pd.read_csv(DATA_BASE + "milk_powder_raw_single.csv")
x = raw["raman_shift_cm-1"].values
y = raw["intensity"].values
spectrum = rp.Spectrum(y, x)

rp.plot.spectra(spectrum, title="① 原始光譜")
rp.plot.show()

## 步驟 1｜裁切 Cropper — 只留下有用的區段

**解決什麼**：光譜兩端常常只有雜訊或濾片造成的假訊號。

**食品分析的慣例**：留 **400–1800 cm⁻¹**（指紋區）。若要看油脂的 C–H 伸縮則另外看 2800–3000。

In [ ]:
cropper = rp.preprocessing.misc.Cropper(region=(450, 1800))
s1 = cropper.apply(spectrum)

rp.plot.spectra(s1, title="② 裁切後")
rp.plot.show()
print("資料點從", len(spectrum.spectral_data), "變成", len(s1.spectral_data))

## 步驟 2｜去尖峰 WhitakerHayes — 清掉宇宙射線

**解決什麼**：宇宙射線造成的假峰。

**原理（一句話）**：真的拉曼峰有寬度，假的尖峰只有 1–2 點寬。演算法看「相鄰點的差值」異常大就判定為尖峰，用鄰居的值補回去。

**做過頭會怎樣**：門檻設太嚴格，真的窄峰也會被當成尖峰刪掉。

In [ ]:
despiker = rp.preprocessing.despike.WhitakerHayes()
s2 = despiker.apply(s1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(s1.spectral_axis, s1.spectral_data); axes[0].set_title("去尖峰前")
axes[1].plot(s2.spectral_axis, s2.spectral_data); axes[1].set_title("去尖峰後")
plt.show()

## 步驟 3｜平滑 SavGol — 壓掉雜訊

**解決什麼**：毛毛的隨機雜訊。

**兩個參數**：
- `window_length`：一次看幾個點來平均（必須是**奇數**）
- `polyorder`：用幾次多項式去擬合（通常 2 或 3）

**做過頭會怎樣**：`window_length` 開太大 → **窄峰被抹平**。下面就是這個災難的現場。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, wl in zip(axes, [9, 31, 71]):
    s = rp.preprocessing.denoise.SavGol(window_length=wl, polyorder=3).apply(s2)
    ax.plot(s.spectral_axis, s.spectral_data)
    ax.set_title(f"window_length = {wl}")
    ax.set_xlim(600, 1200)
plt.suptitle("平滑視窗越大，峰越矮、越胖 —— 訊號被自己毀掉")
plt.show()

s3 = rp.preprocessing.denoise.SavGol(window_length=9, polyorder=3).apply(s2)   # 採用 9

### 👉 換你做

把 `WL` 改成 5、15、41、101，觀察 1085 cm⁻¹ 的乳糖峰高度如何變化。

**思考**：如果你的目標是偵測一個很窄的摻偽物峰，平滑視窗該大還小？

In [ ]:
WL = 9      # 👉 換你做

s_test = rp.preprocessing.denoise.SavGol(window_length=WL, polyorder=3).apply(s2)
i = np.argmin(abs(s_test.spectral_axis - 1085))
print(f"window_length={WL} → 1085 cm-1 峰高 = {s_test.spectral_data[i]:.3f}")

## 步驟 4｜基線校正 — 拿掉螢光背景（最關鍵的一步）

**解決什麼**：螢光造成的大駝峰。

**原理（一句話）**：演算法反覆猜一條「只走在光譜底下」的平滑曲線當作背景，再把它減掉。

RamanSPy 提供十幾種方法，食品樣品常用：

| 方法 | 特性 | 適用 |
|---|---|---|
| `IModPoly()` | 多項式迭代，穩定、快 | 一般食品樣品（本課程預設）|
| `ASPLS()` / `ASLS()` | 懲罰最小平方，彈性大 | 背景形狀複雜時 |
| `AIRPLS()` | 自適應加權 | 螢光極強時 |

**做過頭會怎樣**：基線抓太緊 → 把寬的真實峰也一起減掉（例如蛋白質的醯胺 I 帶）。

In [ ]:
from ramanspy.preprocessing import baseline

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, (name, method) in zip(axes, [("IModPoly", baseline.IModPoly()),
                                     ("ASPLS", baseline.ASPLS()),
                                     ("AIRPLS", baseline.AIRPLS())]):
    s = method.apply(s3)
    ax.plot(s.spectral_axis, s.spectral_data)
    ax.set_title(name)
plt.suptitle("不同基線校正方法的結果比較")
plt.show()

s4 = baseline.IModPoly().apply(s3)

## 步驟 5｜歸一化 — 讓不同樣品可以互相比較

**解決什麼**：Y 軸是任意單位。雷射功率變一點，整條光譜就整體變高變矮。

| 方法 | 做法 | 什麼時候用 |
|---|---|---|
| `MinMax()` | 壓到 0–1 之間 | 最直覺，畫圖比較用 |
| `Vector()` | 向量長度 = 1 | 做 PCA / PLS 前的標準做法 |
| `MaxIntensity()` | 除以最大值 | 有明確參考峰時 |

⚠️ **注意**：`AUC()`（面積歸一化）在新版 NumPy（2.0 以上）會報 `np.trapz` 錯誤，本課程避開不用。

In [ ]:
s5 = rp.preprocessing.normalise.MinMax().apply(s4)

rp.plot.spectra(s5, title="⑥ 前處理完成")
rp.plot.show()
print("強度範圍：", round(s5.spectral_data.min(), 3), "~", round(s5.spectral_data.max(), 3))

## 把五步驟串成一條生產線 Pipeline

上面我們一步一步做，是為了讓你看懂。實務上一行搞定，而且**順序很重要**：

```
裁切 → 去尖峰 → 平滑 → 基線校正 → 歸一化
```

**為什麼是這個順序？**
- 去尖峰要在平滑**之前**：否則尖峰會被抹開變成一個小丘，反而更難刪。
- 歸一化一定放**最後**：否則後面的步驟又會改變強度尺度。

In [ ]:
# 本課程統一使用的標準前處理流程
pipeline = rp.preprocessing.Pipeline([
    rp.preprocessing.misc.Cropper(region=(450, 1800)),          # 裁切
    rp.preprocessing.despike.WhitakerHayes(),                    # 去宇宙射線
    rp.preprocessing.denoise.SavGol(window_length=9, polyorder=3),  # 平滑
    rp.preprocessing.baseline.IModPoly(),                        # 基線校正
    rp.preprocessing.normalise.MinMax(),                         # 歸一化
])

result = pipeline.apply(spectrum)
rp.plot.spectra(result, title="Pipeline 一行完成")
rp.plot.show()

## 一次處理 60 個樣品

同一條 pipeline 可以直接套在整批資料上，不用寫迴圈。

In [ ]:
meta, spectra = load_spectra("milk_powder_melamine.csv")
print("原始資料：", spectra.spectral_data.shape, "→ (樣品數, 波數點數)")

processed = pipeline.apply(spectra)
print("處理後  ：", processed.spectral_data.shape)

rp.plot.mean_spectra(processed, title="60 個奶粉樣品的平均光譜 ± 分佈")
rp.plot.show()

### 🧪 自我檢核

1. 為什麼「去尖峰」一定要排在「平滑」前面？
2. 為什麼「歸一化」一定要排在最後？
3. 你要偵測一個半高寬只有 8 cm⁻¹ 的窄峰，`SavGol(window_length=71)` 合適嗎？
4. 基線校正做過頭，最可能誤傷哪一種峰？

<details><summary>▶ 點開看參考答案</summary>

1. 平滑會把 1–2 點寬的尖峰抹成一個小丘，形狀變得像真的峰，之後就刪不掉了。
2. 歸一化是把強度尺度定下來；如果後面還做基線校正或平滑，尺度又會被改變，等於白做。
3. 不合適。資料點間隔 2 cm⁻¹ 時，71 點 = 142 cm⁻¹ 的視窗，遠寬於 8 cm⁻¹ 的峰，峰會被完全抹平。應該用 5–11。
4. 寬帶的真實峰，例如蛋白質的醯胺 I 帶（約 1655 cm⁻¹）、水的寬帶。它們形狀跟螢光背景相似，容易被當成背景減掉。

</details>


---
### 📚 這一本用到的資料與文獻

**資料**：`data/` 內的光譜為**依文獻峰位建立的模擬資料**（`make_data.py`，亂數種子 20260801），刻意加入螢光背景、宇宙射線與雜訊。可用於教學演練，**不可引用為實驗證據**。

**主要文獻**

- Georgiev, D. et al. *RamanSPy: An Open-Source Python Package for Integrative Raman Spectroscopy Data Analysis*. **Anal. Chem.** 2024, 96(21), 8492–8500. doi:10.1021/acs.analchem.4c00383
- Gill, D.; Kilponen, R. G.; Rimai, L. *Resonance Raman Scattering … in Intact Plant Tissues*. **Nature** 1970, 227, 743–744. doi:10.1038/227743a0
- Lu, L. et al. *Resonance Raman scattering of β-carotene … second singlet state*. **J. Photochem. Photobiol. B** 2018, 179, 18–22. doi:10.1016/j.jphotobiol.2017.12.022
- Withnall, R. et al. *Raman spectra of carotenoids in natural products*. **Spectrochim. Acta A** 2003, 59(10), 2207–2212. doi:10.1016/S1386-1425(03)00064-7
- de Oliveira, V. E. et al. *Carotenes and carotenoids in natural biological samples*. **J. Raman Spectrosc.** 2010, 41(6), 642–650. doi:10.1002/jrs.2493
- Portarena, S. et al. *Cultivar discrimination, fatty acid profile and carotenoid characterization of monovarietal olive oils by Raman spectroscopy at a single glance*. **Food Control** 2019, 96, 137–145. doi:10.1016/j.foodcont.2018.09.011
- Chen, Y. et al. *Quantitative analysis of β-carotene and unsaturated fatty acids in blended olive oil via Raman spectroscopy combined with model prediction*. **Food Chemistry** 2025, 470, 142621. doi:10.1016/j.foodchem.2024.142621
- Schmidt, W. et al. *Continuous Temperature-Dependent Raman Spectroscopy of Melamine and Structural Analog Detection in Milk Powder*. **Appl. Spectrosc.** 2015, 69(3), 398–406. doi:10.1366/14-07600
- Zhang, X. et al. *Detection of melamine in liquid milk using SERS*. **J. Raman Spectrosc.** 2010, 41(12), 1655–1660. doi:10.1002/jrs.2629
- Kim, A. et al. *Melamine Sensing in Milk Products by Using SERS*. **Anal. Chem.** 2012, 84(21), 9303–9309. doi:10.1021/ac302025q
- FAO/WHO Codex Alimentarius. *General Standard for Contaminants and Toxins in Food and Feed*, **CXS 193-1995**.

完整清單見課程網站的「數據來源」與「參考文獻」兩節。